# Fraud Detection Project

## Objective
Build and compare machine learning models to detect fraudulent credit card transactions in a highly imbalanced dataset.

## Context
Fraud detection is a binary classification problem where the positive class (fraud) is extremely rare (~0.17%). Two key challenges:
- Standard accuracy is misleading (predicting no fraud gives 99.83% accuracy)
- Models ignore the minority class without special treatment

**Key metrics:** Precision, Recall, F1, PR AUC (most informative for imbalanced data), ROC AUC

## 1. Setup

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib scikit-learn imbalanced-learn xgboost --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_curve, RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42

## 2. Exploratory Data Analysis

In [ ]:
df = pd.read_csv("creditcard.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()
print("\nMissing values:", df.isnull().sum().sum())

In [ ]:
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100

print("Class distribution:")
print(pd.DataFrame({"count": class_counts, "percent": class_pct.round(3)}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(["Legit (0)", "Fraud (1)"], class_counts.values, color=["steelblue", "crimson"])
axes[0].set_title("Class Distribution (count)")
axes[0].set_ylabel("Transactions")
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 1000, str(v), ha="center", fontweight="bold")

axes[1].pie(class_counts.values, labels=["Legit", "Fraud"],
            colors=["steelblue", "crimson"], autopct="%1.2f%%", startangle=90)
axes[1].set_title("Class Distribution (%)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, color, name in [(0, "steelblue", "Legit"), (1, "crimson", "Fraud")]:
    subset = df[df["Class"] == label]["Amount"]
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=name)
axes[0].set_title("Transaction Amount Distribution")
axes[0].set_xlabel("Amount")
axes[0].legend()

df.groupby("Class")["Amount"].describe().T.plot(kind="bar", ax=axes[1])
axes[1].set_title("Amount Statistics by Class")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

print(df.groupby("Class")["Amount"].describe())

## 3. Data Preparation

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train size: {X_train.shape[0]:,} | Fraud in train: {y_train.sum()}")
print(f"Test size:  {X_test.shape[0]:,} | Fraud in test:  {y_test.sum()}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# SMOTE applied only on train data to avoid data leakage
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("After SMOTE:")
print(pd.Series(y_train_smote).value_counts())

## 4. Model Training

All three models are trained on the SMOTE-balanced dataset and evaluated on the **original unbalanced test set** - this reflects real-world conditions.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_smote, y_train_smote)
print("Logistic Regression trained.")

In [ ]:
rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train_smote, y_train_smote)
print("Random Forest trained.")

In [ ]:
xgb = XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE)
xgb.fit(X_train_smote, y_train_smote)
print("XGBoost trained.")

## 5. Model Evaluation

In [ ]:
def evaluate(model, name, X_test, y_test, threshold=0.5):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    sep = "=" * 52
    print(sep)
    print(f"  {name}  (threshold={threshold})")
    print(sep)
    print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print("Confusion Matrix:")
    print(f"  True Negatives  (Legit correctly classified): {tn:,}")
    print(f"  False Positives (Legit blocked as fraud):     {fp:,}")
    print(f"  False Negatives (Fraud missed):               {fn}")
    print(f"  True Positives  (Fraud correctly caught):     {tp}")
    print(f"\n  ROC AUC: {roc_auc_score(y_test, y_prob):.4f}")
    print(f"  PR AUC:  {average_precision_score(y_test, y_prob):.4f}")
    print()

In [ ]:
evaluate(lr,  "Logistic Regression", X_test_scaled, y_test)
evaluate(rf,  "Random Forest",       X_test_scaled, y_test)
evaluate(xgb, "XGBoost",             X_test_scaled, y_test)

### Note on Logistic Regression

LR after SMOTE achieves high recall (~92%) but very low precision (~6%), generating a large number of **false positives**: legitimate transactions flagged as fraud. In practice, this means blocking hundreds of valid customers per day - unacceptable for most financial services.

XGBoost achieves the best trade-off: **high recall** (catching most fraud) with **acceptable precision** (few false positives).

## 6. Precision-Recall & ROC Curves

In [ ]:
models = {
    "Logistic Regression": lr,
    "Random Forest": rf,
    "XGBoost": xgb
}
colors = ["royalblue", "forestgreen", "crimson"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR Curves
for (name, model), color in zip(models.items(), colors):
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    auc = average_precision_score(y_test, y_prob)
    axes[0].plot(recall, precision, label=f"{name} (PR AUC={auc:.3f})", color=color)

axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curve")
axes[0].legend(loc="upper right")
axes[0].grid(alpha=0.3)

# ROC Curves
for (name, model), color in zip(models.items(), colors):
    RocCurveDisplay.from_estimator(
        model, X_test_scaled, y_test,
        name=name, ax=axes[1], color=color
    )

axes[1].set_title("ROC Curve")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Feature Importance (XGBoost)

In [ ]:
feature_names = X.columns.tolist()
importances = xgb.feature_importances_

fi_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False).head(15)

plt.figure(figsize=(10, 5))
plt.barh(fi_df["feature"][::-1], fi_df["importance"][::-1], color="crimson")
plt.xlabel("Importance")
plt.title("XGBoost - Top 15 Most Important Features")
plt.tight_layout()
plt.show()

print(fi_df.to_string(index=False))

## 8. Threshold Tuning

The default threshold is 0.5. Adjusting it allows trading Recall vs Precision based on business priorities:
- **Lower threshold** -> catch more fraud (higher Recall), more false positives
- **Higher threshold** -> fewer false positives (higher Precision), more missed fraud

In [ ]:
y_prob_xgb = xgb.predict_proba(X_test_scaled)[:, 1]
precision_vals, recall_vals, thresholds = precision_recall_curve(y_test, y_prob_xgb)

f1_vals = 2 * (precision_vals[:-1] * recall_vals[:-1]) / (
    precision_vals[:-1] + recall_vals[:-1] + 1e-9
)
best_idx = f1_vals.argmax()
best_threshold = thresholds[best_idx]

plt.figure(figsize=(10, 4))
plt.plot(thresholds, precision_vals[:-1], label="Precision", color="royalblue")
plt.plot(thresholds, recall_vals[:-1], label="Recall", color="crimson")
plt.plot(thresholds, f1_vals, label="F1", color="forestgreen", linestyle="--")
plt.axvline(best_threshold, color='gray', linestyle=':',
            label=f"Best F1 threshold = {best_threshold:.2f}")
plt.xlabel("Threshold")
plt.title("XGBoost - Precision / Recall / F1 vs Threshold")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best F1 threshold: {best_threshold:.2f} -> F1={f1_vals[best_idx]:.3f}")

In [ ]:
for threshold in [0.3, 0.5, round(best_threshold, 2)]:
    evaluate(xgb, "XGBoost", X_test_scaled, y_test, threshold=threshold)

## 10. Cross-Validation (XGBoost)

A single train/test split can be misleading — especially with only 98 fraud cases in the test set. Cross-validation runs the evaluation multiple times on different splits and reports the average, giving a more reliable estimate of real-world performance.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline as ImbPipeline

# Pipeline: SMOTE + XGBoost (prevents leakage across CV folds)
cv_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = cross_validate(
    cv_pipeline, X_train_scaled, y_train,
    cv=cv,
    scoring=["average_precision", "recall", "precision", "f1"],
    n_jobs=-1
)

print("XGBoost 5-Fold Cross-Validation Results:")
print(f"  PR AUC:    {cv_results['test_average_precision'].mean():.3f} (+/- {cv_results['test_average_precision'].std():.3f})")
print(f"  Recall:    {cv_results['test_recall'].mean():.3f} (+/- {cv_results['test_recall'].std():.3f})")
print(f"  Precision: {cv_results['test_precision'].mean():.3f} (+/- {cv_results['test_precision'].std():.3f})")
print(f"  F1:        {cv_results['test_f1'].mean():.3f} (+/- {cv_results['test_f1'].std():.3f})")

## 11. Saving the Model

Serializing the trained model and scaler allows deploying them in a production system without retraining — the model can be loaded and used to score new transactions in real time.

In [ ]:
import joblib

joblib.dump(xgb, "fraud_model.pkl")
joblib.dump(scaler, "scaler.pkl")
print("Model and scaler saved.")

# Example: loading and scoring a new transaction
loaded_model = joblib.load("fraud_model.pkl")
loaded_scaler = joblib.load("scaler.pkl")

sample = X_test.iloc[[0]]
sample_scaled = loaded_scaler.transform(sample)
prob = loaded_model.predict_proba(sample_scaled)[0, 1]
print(f"Fraud probability for sample transaction: {prob:.4f}")

## 12. Conclusion

### Results Summary

| Model | Precision (Fraud) | Recall (Fraud) | PR AUC | False Positives |
|---|---|---|---|---|
| Logistic Regression | ~0.06 | ~0.92 | ~0.72 | ~1,467 |
| Random Forest | ~0.42 | ~0.86 | ~0.80 | ~117 |
| **XGBoost** | **~0.73** | **~0.85** | **~0.86** | **~31** |

### Key Findings

1. **Accuracy is misleading** - all models show >99% accuracy, but irrelevant for imbalanced data
2. **SMOTE significantly improved recall** - allowing models to detect most fraudulent transactions
3. **LR has unacceptable false positive rate** - ~1,467 legitimate customers wrongly blocked per ~56k transactions
4. **XGBoost achieves the best trade-off** - high recall with low false positives
5. **PR AUC is the right metric** - captures minority class performance better than ROC AUC

### Business Interpretation

The XGBoost model at threshold 0.5 catches ~85% of all fraudulent transactions while wrongly blocking only ~31 legitimate customers. The threshold can be adjusted based on business priorities:
- **Risk-averse**: lower threshold to maximize fraud detection (higher recall)
- **Customer-first**: higher threshold to minimize false positives (higher precision)

### Future Improvements

- Hyperparameter tuning (Optuna / GridSearchCV)
- Cross-validation instead of a single split (small fraud test set = high variance)
- Real-time scoring pipeline with model serialization
- Model monitoring and drift detection in production
- Anomaly detection approaches (Isolation Forest, Autoencoders) as complementary signals